# 02 RQ2: longshot bias and liquidity

Secondary test of RQ2 (`docs/hypotheses.md` Section 4), liquidity as defined in
Section 3 and clarified in D2.

- **Longshot:** in each 1-day snapshot, the cheaper side when it is priced below
  $0.10 (side-agnostic, T6). Its outcome is whether that side won.
- **H2a:** mean(won − price) over longshot snapshots is negative. Two-sided, with
  inference by cluster (D8.1).
- **H2b:** in an OLS regression of (won − price) on log liquidity with format fixed
  effects and cluster-robust SEs, the liquidity coefficient is positive. Two-sided.
- α = 0.05 per hypothesis, Holm across H2a and H2b. Other horizons are descriptive.

The estimation details and the RQ2 sensitivities are fixed in D8 (Section 9,
c34c157). `RUN_CONFIRMATORY` controls whether the outcome column is loaded at all;
with it False, only the planted control on simulated outcomes runs.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests

RUN_CONFIRMATORY = True    # False: outcome column never loaded (planted control only)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

SEED = 20261112          # D7.5, reused
B = 1000                 # bootstrap resamples (Section 4)
LONGSHOT = 0.10          # Section 4
PRIMARY_H = "1d"         # Section 4

pd.set_option("display.width", 200)

cols = ["market_id", "horizon", "snap_price", "is_primary", "within_cap", "volume_usd",
        "liquidity_usd", "format", "subject", "cluster_id", "event_id", "close_year",
        "early_close"] + (["token1_won"] if RUN_CONFIRMATORY else [])
frame = pd.read_parquet(DATA / "mart_analysis.parquet", columns=cols)
vwap = pd.read_parquet(DATA / "mart_vwap.parquet")[["market_id", "horizon", "vwap_price"]]
frame = frame.merge(vwap, on=["market_id", "horizon"], how="left", validate="one_to_one")


def longshots(df, price_col="snap_price"):
    """Cheaper side (T6) when priced below LONGSHOT: token1 if p < 0.5, else token2 at 1 - p."""
    d = df[df[price_col].notna()].copy()
    d["ls_is_token1"] = d[price_col] < 0.5
    d["ls_price"] = np.where(d["ls_is_token1"], d[price_col], 1 - d[price_col])
    d["log_liq"] = np.log1p(d["liquidity_usd"])       # P3
    d["template"] = (d["format"] == "template").astype(float)
    if RUN_CONFIRMATORY:
        d["won"] = np.where(d["ls_is_token1"], d["token1_won"], ~d["token1_won"]).astype(float)
    return d[d["ls_price"] < LONGSHOT].reset_index(drop=True)


primary = frame[frame["is_primary"]]
ls = longshots(primary)
ls1d = ls[ls["horizon"] == PRIMARY_H].reset_index(drop=True)
ls.groupby(["horizon", "format"]).agg(
    snapshots=("ls_price", "size"), clusters=("cluster_id", "nunique"),
    zero_liquidity=("liquidity_usd", lambda s: int((s <= 0).sum())),
    median_price=("ls_price", "median"),
)

snapshots  clusters  zero_liquidity  median_price
horizon format                                                     
1d      event         50466     10563            1707         0.020
        template      93954       529             418         0.010
1h      event        186184      9616             143         0.001
        template      75241       485              44         0.001
1w      event         30345      8315             584         0.016
        template       9253       266             919         0.020

## 1. Estimation details (D8)

Section 4 fixes the hypotheses; D8 fixes these choices, mirroring D7 for RQ1. None
depends on an outcome.

- **P1. H2a test.** The snapshot-level mean of (won − price) over the 1-day longshot
  snapshots, both formats pooled: an intercept-only OLS with CR1 cluster-robust SEs on
  the Section 3 clusters, z reference distribution, two-sided, 95% Wald CI. This is
  the mean the hypothesis names, with inference by cluster. Weighting each cluster
  equally instead would give the 4.7% of clusters that are template series (65% of
  snapshots) under 5% of the weight; that unweighted cluster-mean t test is reported
  as a descriptive.
- **P2. H2b model.** OLS of (won − price) on log liquidity and a template indicator
  (format fixed effect), CR1 cluster-robust SEs on the Section 3 clusters, z
  reference distribution (as D7.1), two-sided, 95% Wald CI.
- **P3. Zero liquidity.** Liquidity is log(1 + USD), so a snapshot that is the
  market's first trade (liquidity 0) stays in the sample at 0.
- **P4. Multiplicity.** Holm across H2a and H2b at family-wise α = 0.05.
- **P5. Descriptives.** For every horizon × format: snapshot count, clusters, mean
  (won − price) with a 1,000-resample cluster-bootstrap percentile CI (seed as D7.5).
  At 1 day: the same by liquidity quintile and by subject, plus the unweighted
  cluster-mean t test from P1.
- **P6. Sensitivities.** Fixed in D8.6; the cell is not written yet, and the gate stays
  closed until it is.

In [2]:
def h2a(d, y, cluster_col="cluster_id"):
    """P1: snapshot mean of (y - price), intercept-only OLS, CR1 cluster SEs, z."""
    g = pd.factorize(d[cluster_col])[0]
    res = sm.OLS(y - d["ls_price"].to_numpy(), np.ones(len(d))).fit(
        cov_type="cluster", cov_kwds={"groups": g})
    b, se = res.params[0], res.bse[0]
    return dict(test="H2a", estimate=b, se=se, ci_low=b - 1.959964 * se,
                ci_high=b + 1.959964 * se, p_value=res.pvalues[0], clusters=g.max() + 1, n=len(d))


def h2a_cluster_means(d, y):
    """P5 descriptive: one-sample t test on unweighted cluster means of (y - price)."""
    g = pd.factorize(d["cluster_id"])[0]
    m = np.bincount(g, y - d["ls_price"].to_numpy()) / np.bincount(g)
    G = len(m)
    se = m.std(ddof=1) / np.sqrt(G)
    q = stats.t.ppf(0.975, G - 1)
    return dict(estimate=m.mean(), se=se, ci_low=m.mean() - q * se, ci_high=m.mean() + q * se,
                p_value=2 * stats.t.sf(abs(m.mean() / se), G - 1), clusters=G, n=len(d))


def h2b(d, y, cluster_col="cluster_id"):
    """P2: OLS of (y - price) on log liquidity + format FE, CR1 cluster SEs, z."""
    X = np.column_stack([np.ones(len(d)), d["log_liq"].to_numpy(), d["template"].to_numpy()])
    g = pd.factorize(d[cluster_col])[0]
    res = sm.OLS(y - d["ls_price"].to_numpy(), X).fit(cov_type="cluster", cov_kwds={"groups": g})
    b, se = res.params[1], res.bse[1]
    return dict(test="H2b", estimate=b, se=se, ci_low=b - 1.959964 * se,
                ci_high=b + 1.959964 * se, p_value=res.pvalues[1],
                clusters=g.max() + 1, n=len(d))


def ols_truth(d, q):
    """Population H2b coefficient: OLS of the true (q - price) on the same design."""
    X = np.column_stack([np.ones(len(d)), d["log_liq"].to_numpy(), d["template"].to_numpy()])
    return np.linalg.lstsq(X, q - d["ls_price"].to_numpy(), rcond=None)[0][1]


def h2a_truth(d, q):
    return (q - d["ls_price"].to_numpy()).mean()

## 2. Planted control (simulated outcomes only)

The real prices, liquidity, formats and clusters of the 1-day longshot frame are kept;
only the outcome is simulated, so the check exercises the exact sample and cluster
structure the test will see.

- **Planted:** true win probability q = price × (1 − k), with
  k = clip(0.30 − 0.05 × (log liquidity − median), 0, 1). Longshots are overpriced by
  30% at median liquidity, less so in deeper markets: H2a's truth is negative and
  H2b's truth is positive.
- **Null:** q = price, so both truths are 0.

Each scenario is drawn 50 times. The truths are computed from q itself: H2a's is the
snapshot mean of (q − price), H2b's is the OLS coefficient of (q − price) on
the same design (E[won | X] = q, so OLS on simulated outcomes is unbiased for it).
Pass: 95% CIs cover the truth in roughly 95% of draws in both scenarios, the planted
effects are rejected in most draws, and the null rejection rate is near 5%.

In [3]:
rng = np.random.default_rng(SEED)
R = 50
k = np.clip(0.30 - 0.05 * (ls1d["log_liq"] - ls1d["log_liq"].median()), 0, 1).to_numpy()
scenarios = {"planted": ls1d["ls_price"].to_numpy() * (1 - k), "null": ls1d["ls_price"].to_numpy()}

rows = []
for name, q in scenarios.items():
    truth = {"H2a": h2a_truth(ls1d, q), "H2b": ols_truth(ls1d, q)}
    for r in range(R):
        y = rng.binomial(1, q).astype(float)
        for fit in (h2a(ls1d, y), h2b(ls1d, y)):
            rows.append({"scenario": name, "rep": r, "truth": truth[fit["test"]], **fit})
pc = pd.DataFrame(rows)
pc["covers"] = (pc["ci_low"] <= pc["truth"]) & (pc["truth"] <= pc["ci_high"])
pc["rejects_0"] = pc["p_value"] < 0.05
summary = pc.groupby(["scenario", "test"]).agg(
    truth=("truth", "first"), mean_estimate=("estimate", "mean"), mean_se=("se", "mean"),
    sd_estimate=("estimate", "std"), coverage=("covers", "mean"), reject_rate=("rejects_0", "mean"),
)
summary.to_csv(RESULTS / "rq2_planted_control.csv")
summary

truth  mean_estimate   mean_se  sd_estimate  coverage  reject_rate
scenario test                                                                       
null     H2a   0.000000       0.000065  0.000392     0.000433      0.94         0.06
         H2b   0.000000      -0.000021  0.000137     0.000130      0.98         0.02
planted  H2a  -0.007591      -0.007667  0.000476     0.000359      0.96         1.00
         H2b   0.001626       0.001648  0.000133     0.000105      0.98         1.00

In [4]:
checks = {
    "coverage >= 0.88 in every scenario x test": bool((summary["coverage"] >= 0.88).all()),
    "planted effects rejected in >= 80% of draws": bool((summary.loc["planted", "reject_rate"] >= 0.80).all()),
    "planted signs right (H2a < 0, H2b > 0)": bool(summary.loc[("planted", "H2a"), "mean_estimate"] < 0
                                                   and summary.loc[("planted", "H2b"), "mean_estimate"] > 0),
    "null rejection rate <= 0.12": bool((summary.loc["null", "reject_rate"] <= 0.12).all()),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
PLANTED_CONTROL_PASSED = all(checks.values())

PASS coverage >= 0.88 in every scenario x test
PASS planted effects rejected in >= 80% of draws
PASS planted signs right (H2a < 0, H2b > 0)
PASS null rejection rate <= 0.12


## 3. Confirmatory tests (gated)

Runs only with `RUN_CONFIRMATORY = True` and a passing planted control. H2a and H2b at
the 1-day horizon, Holm across the two.

In [5]:
if RUN_CONFIRMATORY and PLANTED_CONTROL_PASSED:
    y = ls1d["won"].to_numpy()
    tests = pd.DataFrame([h2a(ls1d, y), h2b(ls1d, y)])
    tests["p_holm"] = multipletests(tests["p_value"], method="holm")[1]
    tests["significant_holm"] = tests["p_holm"] < 0.05
    tests.to_csv(RESULTS / "rq2_tests.csv", index=False)
    display(tests.round(5))
else:
    print("confirmatory tests skipped (gate closed)")

,test,estimate,se,ci_low,ci_high,p_value,clusters,n,p_holm,significant_holm
0,H2a,-0.00247,0.00056,-0.00357,-0.00137,0.00001,11067,144420,0.00002,True
1,H2b,-0.00033,0.00020,-0.00072,0.00005,0.08826,11067,144420,0.08826,False


## 4. Descriptives (gated)

P5: mean(won − price) per horizon × format, and at 1 day by liquidity quintile and by
subject, with cluster-bootstrap 95% CIs; then the unweighted cluster-mean t test.

In [6]:
def boot_mean(d, B=B):
    resid = d["won"].to_numpy() - d["ls_price"].to_numpy()
    g, _ = pd.factorize(d["cluster_id"])
    C = g.max() + 1
    W = rng_desc.multinomial(C, np.full(C, 1 / C), size=B).astype(float)
    b = (W @ np.bincount(g, resid, C)) / (W @ np.bincount(g, minlength=C))
    lo, hi = np.percentile(b, [2.5, 97.5])
    return dict(n=len(d), clusters=C, mean_resid=resid.mean(), lo=lo, hi=hi)


if RUN_CONFIRMATORY:
    rng_desc = np.random.default_rng(SEED)
    desc = pd.DataFrame([{"horizon": h, "format": f, **boot_mean(d)}
                         for (h, f), d in ls.groupby(["horizon", "format"])])
    desc.to_csv(RESULTS / "rq2_descriptives.csv", index=False)
    q = ls1d.assign(liq_quintile=pd.qcut(ls1d["log_liq"], 5, labels=False, duplicates="drop") + 1)
    quint = pd.DataFrame([{"liq_quintile": k, "median_liq_usd": d["liquidity_usd"].median(), **boot_mean(d)}
                          for k, d in q.groupby("liq_quintile")])
    quint.to_csv(RESULTS / "rq2_liquidity_quintiles.csv", index=False)
    subj = pd.DataFrame([{"subject": s, **boot_mean(d)} for s, d in ls1d.groupby("subject")])
    subj.to_csv(RESULTS / "rq2_by_subject.csv", index=False)
    cm = pd.DataFrame([h2a_cluster_means(ls1d, ls1d["won"].to_numpy())])
    cm.to_csv(RESULTS / "rq2_h2a_cluster_means.csv", index=False)
    display(desc.round(5), quint.round(5), subj.round(5), cm.round(5))
else:
    print("descriptives skipped (gate closed)")

,horizon,format,n,clusters,mean_resid,lo,hi
0,1d,event,50466,10563,-0.00010,-0.00220,0.00260
1,1d,template,93954,529,-0.00374,-0.00479,-0.00256
2,1h,event,186184,9616,-0.00183,-0.00227,-0.00149
3,1h,template,75241,485,-0.00250,-0.00313,-0.00172
4,1w,event,30345,8315,0.00050,-0.00200,0.00334
5,1w,template,9253,266,-0.00098,-0.00769,0.00375


,liq_quintile,median_liq_usd,n,clusters,mean_resid,lo,hi
0,1,15.280,28886,1915,0.00039,-0.00197,0.00314
1,2,176.850,28882,2094,-0.00462,-0.00619,-0.00280
2,3,597.750,28884,2690,-0.00507,-0.00704,-0.00314
3,4,2493.205,28884,4397,-0.00162,-0.00347,0.00039
4,5,30900.050,28884,6353,-0.00142,-0.00324,0.00063


,subject,n,clusters,mean_resid,lo,hi
0,crypto,22851,823,-0.00174,-0.00406,0.00224
1,other,29281,5120,-0.00375,-0.00556,-0.00182
2,politics,10161,3178,-0.00145,-0.00467,0.00187
3,sports,22059,1943,0.00346,-0.00079,0.00963
4,weather,60068,151,-0.00447,-0.00565,-0.00320


,estimate,se,ci_low,ci_high,p_value,clusters,n
0,0.00248,0.00131,-0.00009,0.00506,0.05846,11067,144420


## 5. Sensitivities (Section 5, D8.6; gated)

H2a and H2b are re-estimated at 1 day under each Section 5 variant. Each reports its
coefficient and 95% CI only, with no p-values and no Holm.

- **5.1** drops the volume floor; **5.2** raises it to $10,000.
- **5.3** takes the longshot side and price from the 5-minute VWAP (D7.7). Markets
  with no trade in that window drop out. Liquidity stays the cumulative volume before
  the last-trade snapshot.
- **5.4** drops the staleness cap.
- **5.5** clusters by `event_id`.
- **5.6** fits each `closedTime` year that has at least 30 clusters.
- **5.7** drops early closes. Read it as a fragility check, not a bias correction:
  early close is itself selected on the outcome in deadline markets (notebook 01b).

A variant whose population has only one format has a constant template column, so
the format fixed effect drops out of H2b for it.

In [7]:
if RUN_CONFIRMATORY:
    day = lambda df: df[df["horizon"] == PRIMARY_H]
    f = frame
    specs = {
        "5.1 no volume floor": (longshots(day(f[f["within_cap"]])), "cluster_id"),
        "5.2 volume >= $10k": (longshots(day(f[f["within_cap"] & (f["volume_usd"] >= 10_000)])), "cluster_id"),
        "5.3 5-min VWAP": (longshots(day(primary), "vwap_price"), "cluster_id"),
        "5.4 no staleness cap": (longshots(day(f[f["volume_usd"] >= 1000])), "cluster_id"),
        "5.5 event clustering": (ls1d, "event_id"),
        "5.7 no early closes": (ls1d[~ls1d["early_close"].astype(bool)].reset_index(drop=True), "cluster_id"),
    }
    for year in sorted(ls1d["close_year"].unique()):
        specs[f"5.6 year {year}"] = (ls1d[ls1d["close_year"] == year].reset_index(drop=True), "cluster_id")
    rows = []
    for name, (d, cc) in specs.items():
        G = d[cc].nunique()
        for fn, test in ((h2a, "H2a"), (h2b, "H2b")):
            base = {"sensitivity": name, "test": test, "n": len(d), "clusters": G,
                    "formats": d["format"].nunique()}
            if G < 30:
                rows.append({**base, "fitted": False})
                continue
            r = fn(d, d["won"].to_numpy(), cc)
            rows.append({**base, "fitted": True, "estimate": r["estimate"],
                         "ci_low": r["ci_low"], "ci_high": r["ci_high"]})
    sens = pd.DataFrame(rows)
    sens.to_csv(RESULTS / "rq2_sensitivities.csv", index=False)
    display(sens.round(5))
else:
    print("sensitivities skipped (gate closed)")

,sensitivity,test,n,clusters,formats,fitted,estimate,ci_low,ci_high
0,5.1 no volume floor,H2a,174498,11591,2,True,-0.00280,-0.00401,-0.00159
1,5.1 no volume floor,H2b,174498,11591,2,True,-0.00012,-0.00055,0.00030
2,5.2 volume >= $10k,H2a,67255,8641,2,True,0.00432,0.00199,0.00665
3,5.2 volume >= $10k,H2b,67255,8641,2,True,-0.00241,-0.00335,-0.00146
4,5.3 5-min VWAP,H2a,19892,2243,2,True,-0.00033,-0.00253,0.00187
5,5.3 5-min VWAP,H2b,19892,2243,2,True,0.00064,-0.00019,0.00146
6,5.4 no staleness cap,H2a,182918,13397,2,True,-0.00242,-0.00363,-0.00121
7,5.4 no staleness cap,H2b,182918,13397,2,True,-0.00038,-0.00076,0.00001
8,5.5 event clustering,H2a,144420,41336,2,True,-0.00247,-0.00324,-0.00169
9,5.5 event clustering,H2b,144420,41336,2,True,-0.00033,-0.00063,-0.00004
